In [1]:
from deepface import DeepFace
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC  # Support Vector Classifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import tensorflow as tf
import os

# --- 1. Define Paths and Constants ---
DATA_DIR = r"C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\data\imgData\prepImg\step4"
IMG_SIZE = (224, 224) 
BATCH_SIZE = 32
SEED = 123             

# --- 2. Load the Dataset ---
# 80% Training
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

# 20% for Val and Test
remaining_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

# Final split for 10% Test / 10% Val
val_batches = tf.data.experimental.cardinality(remaining_ds)
test_ds = remaining_ds.take(val_batches // 2)
val_ds = remaining_ds.skip(val_batches // 2)

# --- 3. Combined Preprocessing (Normalization + Label Flip) ---
# We do this in ONE step to save computation time.
normalization_layer = tf.keras.layers.Rescaling(1./255)

def preprocess(x, y):
    # Normalize pixels to [0, 1]
    x = normalization_layer(x)
    # Flip labels so Autistic (alphabetically first, so 0) becomes 1
    y = 1 - y
    return x, y

train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)
test_ds = test_ds.map(preprocess)

# --- 4. Performance Optimization ---
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("\n--- Data Preparation Complete ---")
print("Confirmed: Autistic is now labeled 1, Non_Autistic is labeled 0.")
for images, labels in train_ds.take(1):
    print(f"Sample Batch Labels: {labels.numpy().flatten()[:5]}")
    print(f"Pixel Max: {tf.reduce_max(images).numpy()}") # Should be <= 1.0

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from deepface import DeepFace

# --- 1. Load the Pre-trained Backbone ---
vgg_model_wrapper = DeepFace.build_model("VGG-Face")
full_vgg_face = vgg_model_wrapper.model 

# --- 2. Strip away the recognition layers ---
# Based on your error log, 'conv2d_14' is the final feature-rich layer.
backbone_output = full_vgg_face.get_layer("conv2d_14").output
feature_extractor = Model(inputs=full_vgg_face.input, outputs=backbone_output)

# --- 3. Freeze the Backbone Layers ---
feature_extractor.trainable = False

# --- 4. Attach the Custom Head ---
inputs = Input(shape=(224, 224, 3), name="face_input")

# Pass input through frozen backbone
# training=False is critical to keep the model in inference mode
x = feature_extractor(inputs, training=False)

# A. Global Average Pooling 
# This converts the 2D output of conv2d_14 into a 1D vector
x = GlobalAveragePooling2D()(x)

# B. Dense Layer (ASD Feature Learner)
feature_layer = Dense(256, activation='relu', name='asd_feature_vector')(x)

# C. Output Layer (Sigmoid for binary probability)
prediction = Dense(1, activation='sigmoid', name='classification_output')(feature_layer)

# --- 5. Build the Unified Model ---
# This meets your requirement for dual outputs (prediction + feature vector)
unified_model = Model(inputs=inputs, outputs=[prediction, feature_layer])

# Verify
unified_model.summary()
# Adjust datasets to handle the dual-output architecture
def map_for_dual_output(img, label):
    # The first 'label' is for classification_output
    # The second 'None' tells Keras not to calculate loss for the feature_vector output
    return img, {'classification_output': label, 'asd_feature_vector': label}

train_ds_final = train_ds.map(map_for_dual_output)
val_ds_final = val_ds.map(map_for_dual_output)
test_ds_final = test_ds.map(map_for_dual_output)
unified_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss={
        'classification_output': 'binary_crossentropy',
        'asd_feature_vector': None # No loss for the feature vector output
    },
    metrics={
        'classification_output': ['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    }
)
try:
    # Test a single dummy prediction to ensure the "Dual Output" works
    dummy_input = tf.random.uniform((1, 224, 224, 3))
    pred, feat = unified_model.predict(dummy_input)
    print(f"Prediction Output Shape: {pred.shape}") # Should be (1, 1)
    print(f"Feature Vector Shape: {feat.shape}")    # Should be (1, 256)
    print("\n✅ CHECKLIST PASSED: Architecture and Data are aligned.")
except Exception as e:
    print(f"\n❌ CHECKLIST FAILED: {e}")
    # Function to align dataset labels with the multi-output model architecture
def prepare_for_training(img, label):
    # We pass the same label to both output keys so the model knows where to look
    return img, {'classification_output': label, 'asd_feature_vector': label}

# Apply the mapping to all dataset splits
train_ds_final = train_ds.map(prepare_for_training)
val_ds_final = val_ds.map(prepare_for_training)
test_ds_final = test_ds.map(prepare_for_training)

print("Datasets are now configured for Dual-Output Training.")
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 1. Define Callbacks
# Stop if validation loss doesn't improve for 5 consecutive epochs
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True,
    verbose=1
)

# Save the best model weights based on highest validation accuracy
checkpoint_path = r"C:\Users\Yasindu\Desktop\Stuff\1.SLIIT\Research\Project\best_asd_model.h5"
checkpoint = ModelCheckpoint(
    checkpoint_path, 
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max',
    verbose=1
)

# 2. Start the Training Process
# We use 20 epochs, but EarlyStopping will likely halt it earlier if it peaks
print("Starting Training...")
history = unified_model.fit(
    train_ds_final,
    validation_data=val_ds_final,
    epochs=20,
    callbacks=[early_stop, checkpoint],
    verbose=1
)Starting Training...
Epoch 1/20
98/98 [==============================] - 71s 624ms/step - loss: 0.6788 - classification_output_loss: 0.6788 - classification_output_accuracy: 0.6442 - classification_output_precision: 0.7420 - classification_output_recall: 0.4449 - val_loss: 0.6595 - val_classification_output_loss: 0.6595 - val_classification_output_accuracy: 0.6867 - val_classification_output_precision: 0.6954 - val_classification_output_recall: 0.6782
Epoch 2/20
98/98 [==============================] - 46s 471ms/step - loss: 0.6344 - classification_output_loss: 0.6344 - classification_output_accuracy: 0.7116 - classification_output_precision: 0.7295 - classification_output_recall: 0.6747 - val_loss: 0.6174 - val_classification_output_loss: 0.6174 - val_classification_output_accuracy: 0.6967 - val_classification_output_precision: 0.7516 - val_classification_output_recall: 0.5990
Epoch 3/20
98/98 [==============================] - 44s 452ms/step - loss: 0.5919 - classification_output_loss: 0.5919 - classification_output_accuracy: 0.7278 - classification_output_precision: 0.7507 - classification_output_recall: 0.6843 - val_loss: 0.5835 - val_classification_output_loss: 0.5835 - val_classification_output_accuracy: 0.7243 - val_classification_output_precision: 0.7805 - val_classification_output_recall: 0.6337
Epoch 4/20
98/98 [==============================] - 41s 424ms/step - loss: 0.5590 - classification_output_loss: 0.5590 - classification_output_accuracy: 0.7396 - classification_output_precision: 0.7634 - classification_output_recall: 0.6964 - val_loss: 0.5568 - val_classification_output_loss: 0.5568 - val_classification_output_accuracy: 0.7469 - val_classification_output_precision: 0.7617 - val_classification_output_recall: 0.7277
Epoch 5/20
98/98 [==============================] - 45s 456ms/step - loss: 0.5336 - classification_output_loss: 0.5336 - classification_output_accuracy: 0.7514 - classification_output_precision: 0.7769 - classification_output_recall: 0.7072 - val_loss: 0.5371 - val_classification_output_loss: 0.5371 - val_classification_output_accuracy: 0.7544 - val_classification_output_precision: 0.7653 - val_classification_output_recall: 0.7426
Epoch 6/20
98/98 [==============================] - 44s 449ms/step - loss: 0.5133 - classification_output_loss: 0.5133 - classification_output_accuracy: 0.7632 - classification_output_precision: 0.7776 - classification_output_recall: 0.7390 - val_loss: 0.5225 - val_classification_output_loss: 0.5225 - val_classification_output_accuracy: 0.7594 - val_classification_output_precision: 0.7704 - val_classification_output_recall: 0.7475
Epoch 7/20
98/98 [==============================] - 31s 315ms/step - loss: 0.4955 - classification_output_loss: 0.4955 - classification_output_accuracy: 0.7735 - classification_output_precision: 0.7919 - classification_output_recall: 0.7435 - val_loss: 0.5102 - val_classification_output_loss: 0.5102 - val_classification_output_accuracy: 0.7644 - val_classification_output_precision: 0.7700 - val_classification_output_recall: 0.7624
Epoch 8/20
98/98 [==============================] - 29s 299ms/step - loss: 0.4799 - classification_output_loss: 0.4799 - classification_output_accuracy: 0.7884 - classification_output_precision: 0.8047 - classification_output_recall: 0.7632 - val_loss: 0.5010 - val_classification_output_loss: 0.5010 - val_classification_output_accuracy: 0.7694 - val_classification_output_precision: 0.7723 - val_classification_output_recall: 0.7723
Epoch 9/20
98/98 [==============================] - 29s 301ms/step - loss: 0.4665 - classification_output_loss: 0.4665 - classification_output_accuracy: 0.7948 - classification_output_precision: 0.8077 - classification_output_recall: 0.7753 - val_loss: 0.4951 - val_classification_output_loss: 0.4951 - val_classification_output_accuracy: 0.7569 - val_classification_output_precision: 0.7749 - val_classification_output_recall: 0.7327
Epoch 10/20
98/98 [==============================] - 30s 302ms/step - loss: 0.4536 - classification_output_loss: 0.4536 - classification_output_accuracy: 0.8050 - classification_output_precision: 0.8243 - classification_output_recall: 0.7766 - val_loss: 0.4861 - val_classification_output_loss: 0.4861 - val_classification_output_accuracy: 0.7794 - val_classification_output_precision: 0.7822 - val_classification_output_recall: 0.7822
Epoch 11/20
98/98 [==============================] - 30s 305ms/step - loss: 0.4418 - classification_output_loss: 0.4418 - classification_output_accuracy: 0.8095 - classification_output_precision: 0.8282 - classification_output_recall: 0.7823 - val_loss: 0.4798 - val_classification_output_loss: 0.4798 - val_classification_output_accuracy: 0.7995 - val_classification_output_precision: 0.7933 - val_classification_output_recall: 0.8168
Epoch 12/20
98/98 [==============================] - 29s 299ms/step - loss: 0.4312 - classification_output_loss: 0.4312 - classification_output_accuracy: 0.8159 - classification_output_precision: 0.8283 - classification_output_recall: 0.7982 - val_loss: 0.4756 - val_classification_output_loss: 0.4756 - val_classification_output_accuracy: 0.7995 - val_classification_output_precision: 0.7961 - val_classification_output_recall: 0.8119
Epoch 13/20
98/98 [==============================] - 29s 299ms/step - loss: 0.4210 - classification_output_loss: 0.4210 - classification_output_accuracy: 0.8220 - classification_output_precision: 0.8334 - classification_output_recall: 0.8059 - val_loss: 0.4720 - val_classification_output_loss: 0.4720 - val_classification_output_accuracy: 0.7995 - val_classification_output_precision: 0.8020 - val_classification_output_recall: 0.8020
Epoch 14/20
98/98 [==============================] - 29s 299ms/step - loss: 0.4112 - classification_output_loss: 0.4112 - classification_output_accuracy: 0.8290 - classification_output_precision: 0.8411 - classification_output_recall: 0.8122 - val_loss: 0.4676 - val_classification_output_loss: 0.4676 - val_classification_output_accuracy: 0.7995 - val_classification_output_precision: 0.7990 - val_classification_output_recall: 0.8069
Epoch 15/20
98/98 [==============================] - 29s 300ms/step - loss: 0.4020 - classification_output_loss: 0.4020 - classification_output_accuracy: 0.8280 - classification_output_precision: 0.8404 - classification_output_recall: 0.8109 - val_loss: 0.4653 - val_classification_output_loss: 0.4653 - val_classification_output_accuracy: 0.7995 - val_classification_output_precision: 0.7933 - val_classification_output_recall: 0.8168
Epoch 16/20
98/98 [==============================] - 30s 309ms/step - loss: 0.3929 - classification_output_loss: 0.3929 - classification_output_accuracy: 0.8373 - classification_output_precision: 0.8443 - classification_output_recall: 0.8281 - val_loss: 0.4641 - val_classification_output_loss: 0.4641 - val_classification_output_accuracy: 0.7945 - val_classification_output_precision: 0.8093 - val_classification_output_recall: 0.7772
Epoch 17/20
98/98 [==============================] - 29s 301ms/step - loss: 0.3850 - classification_output_loss: 0.3850 - classification_output_accuracy: 0.8360 - classification_output_precision: 0.8443 - classification_output_recall: 0.8250 - val_loss: 0.4640 - val_classification_output_loss: 0.4640 - val_classification_output_accuracy: 0.7845 - val_classification_output_precision: 0.8053 - val_classification_output_recall: 0.7574
Epoch 18/20
98/98 [==============================] - 29s 300ms/step - loss: 0.3785 - classification_output_loss: 0.3785 - classification_output_accuracy: 0.8417 - classification_output_precision: 0.8506 - classification_output_recall: 0.8300 - val_loss: 0.4578 - val_classification_output_loss: 0.4578 - val_classification_output_accuracy: 0.7945 - val_classification_output_precision: 0.8000 - val_classification_output_recall: 0.7921
Epoch 19/20
98/98 [==============================] - 29s 300ms/step - loss: 0.3700 - classification_output_loss: 0.3700 - classification_output_accuracy: 0.8478 - classification_output_precision: 0.8575 - classification_output_recall: 0.8351 - val_loss: 0.4562 - val_classification_output_loss: 0.4562 - val_classification_output_accuracy: 0.7970 - val_classification_output_precision: 0.8071 - val_classification_output_recall: 0.7871
Epoch 20/20
98/98 [==============================] - 30s 302ms/step - loss: 0.3617 - classification_output_loss: 0.3617 - classification_output_accuracy: 0.8539 - classification_output_precision: 0.8683 - classification_output_recall: 0.8351 - val_loss: 0.4535 - val_classification_output_loss: 0.4535 - val_classification_output_accuracy: 0.8045 - val_classification_output_precision: 0.7952 - val_classification_output_recall: 0.8267
Starting Training...
Epoch 1/20
98/98 [==============================] - 71s 624ms/step - loss: 0.6788 - classification_output_loss: 0.6788 - classification_output_accuracy: 0.6442 - classification_output_precision: 0.7420 - classification_output_recall: 0.4449 - val_loss: 0.6595 - val_classification_output_loss: 0.6595 - val_classification_output_accuracy: 0.6867 - val_classification_output_precision: 0.6954 - val_classification_output_recall: 0.6782
Epoch 2/20
98/98 [==============================] - 46s 471ms/step - loss: 0.6344 - classification_output_loss: 0.6344 - classification_output_accuracy: 0.7116 - classification_output_precision: 0.7295 - classification_output_recall: 0.6747 - val_loss: 0.6174 - val_classification_output_loss: 0.6174 - val_classification_output_accuracy: 0.6967 - val_classification_output_precision: 0.7516 - val_classification_output_recall: 0.5990
Epoch 3/20
98/98 [==============================] - 44s 452ms/step - loss: 0.5919 - classification_output_loss: 0.5919 - classification_output_accuracy: 0.7278 - classification_output_precision: 0.7507 - classification_output_recall: 0.6843 - val_loss: 0.5835 - val_classification_output_loss: 0.5835 - val_classification_output_accuracy: 0.7243 - val_classification_output_precision: 0.7805 - val_classification_output_recall: 0.6337
Epoch 4/20
98/98 [==============================] - 41s 424ms/step - loss: 0.5590 - classification_output_loss: 0.5590 - classification_output_accuracy: 0.7396 - classification_output_precision: 0.7634 - classification_output_recall: 0.6964 - val_loss: 0.5568 - val_classification_output_loss: 0.5568 - val_classification_output_accuracy: 0.7469 - val_classification_output_precision: 0.7617 - val_classification_output_recall: 0.7277
Epoch 5/20
98/98 [==============================] - 45s 456ms/step - loss: 0.5336 - classification_output_loss: 0.5336 - classification_output_accuracy: 0.7514 - classification_output_precision: 0.7769 - classification_output_recall: 0.7072 - val_loss: 0.5371 - val_classification_output_loss: 0.5371 - val_classification_output_accuracy: 0.7544 - val_classification_output_precision: 0.7653 - val_classification_output_recall: 0.7426
Epoch 6/20
98/98 [==============================] - 44s 449ms/step - loss: 0.5133 - classification_output_loss: 0.5133 - classification_output_accuracy: 0.7632 - classification_output_precision: 0.7776 - classification_output_recall: 0.7390 - val_loss: 0.5225 - val_classification_output_loss: 0.5225 - val_classification_output_accuracy: 0.7594 - val_classification_output_precision: 0.7704 - val_classification_output_recall: 0.7475
Epoch 7/20
98/98 [==============================] - 31s 315ms/step - loss: 0.4955 - classification_output_loss: 0.4955 - classification_output_accuracy: 0.7735 - classification_output_precision: 0.7919 - classification_output_recall: 0.7435 - val_loss: 0.5102 - val_classification_output_loss: 0.5102 - val_classification_output_accuracy: 0.7644 - val_classification_output_precision: 0.7700 - val_classification_output_recall: 0.7624
Epoch 8/20
98/98 [==============================] - 29s 299ms/step - loss: 0.4799 - classification_output_loss: 0.4799 - classification_output_accuracy: 0.7884 - classification_output_precision: 0.8047 - classification_output_recall: 0.7632 - val_loss: 0.5010 - val_classification_output_loss: 0.5010 - val_classification_output_accuracy: 0.7694 - val_classification_output_precision: 0.7723 - val_classification_output_recall: 0.7723
Epoch 9/20
98/98 [==============================] - 29s 301ms/step - loss: 0.4665 - classification_output_loss: 0.4665 - classification_output_accuracy: 0.7948 - classification_output_precision: 0.8077 - classification_output_recall: 0.7753 - val_loss: 0.4951 - val_classification_output_loss: 0.4951 - val_classification_output_accuracy: 0.7569 - val_classification_output_precision: 0.7749 - val_classification_output_recall: 0.7327
Epoch 10/20
98/98 [==============================] - 30s 302ms/step - loss: 0.4536 - classification_output_loss: 0.4536 - classification_output_accuracy: 0.8050 - classification_output_precision: 0.8243 - classification_output_recall: 0.7766 - val_loss: 0.4861 - val_classification_output_loss: 0.4861 - val_classification_output_accuracy: 0.7794 - val_classification_output_precision: 0.7822 - val_classification_output_recall: 0.7822
Epoch 11/20
98/98 [==============================] - 30s 305ms/step - loss: 0.4418 - classification_output_loss: 0.4418 - classification_output_accuracy: 0.8095 - classification_output_precision: 0.8282 - classification_output_recall: 0.7823 - val_loss: 0.4798 - val_classification_output_loss: 0.4798 - val_classification_output_accuracy: 0.7995 - val_classification_output_precision: 0.7933 - val_classification_output_recall: 0.8168
Epoch 12/20
98/98 [==============================] - 29s 299ms/step - loss: 0.4312 - classification_output_loss: 0.4312 - classification_output_accuracy: 0.8159 - classification_output_precision: 0.8283 - classification_output_recall: 0.7982 - val_loss: 0.4756 - val_classification_output_loss: 0.4756 - val_classification_output_accuracy: 0.7995 - val_classification_output_precision: 0.7961 - val_classification_output_recall: 0.8119
Epoch 13/20
98/98 [==============================] - 29s 299ms/step - loss: 0.4210 - classification_output_loss: 0.4210 - classification_output_accuracy: 0.8220 - classification_output_precision: 0.8334 - classification_output_recall: 0.8059 - val_loss: 0.4720 - val_classification_output_loss: 0.4720 - val_classification_output_accuracy: 0.7995 - val_classification_output_precision: 0.8020 - val_classification_output_recall: 0.8020
Epoch 14/20
98/98 [==============================] - 29s 299ms/step - loss: 0.4112 - classification_output_loss: 0.4112 - classification_output_accuracy: 0.8290 - classification_output_precision: 0.8411 - classification_output_recall: 0.8122 - val_loss: 0.4676 - val_classification_output_loss: 0.4676 - val_classification_output_accuracy: 0.7995 - val_classification_output_precision: 0.7990 - val_classification_output_recall: 0.8069
Epoch 15/20
98/98 [==============================] - 29s 300ms/step - loss: 0.4020 - classification_output_loss: 0.4020 - classification_output_accuracy: 0.8280 - classification_output_precision: 0.8404 - classification_output_recall: 0.8109 - val_loss: 0.4653 - val_classification_output_loss: 0.4653 - val_classification_output_accuracy: 0.7995 - val_classification_output_precision: 0.7933 - val_classification_output_recall: 0.8168
Epoch 16/20
98/98 [==============================] - 30s 309ms/step - loss: 0.3929 - classification_output_loss: 0.3929 - classification_output_accuracy: 0.8373 - classification_output_precision: 0.8443 - classification_output_recall: 0.8281 - val_loss: 0.4641 - val_classification_output_loss: 0.4641 - val_classification_output_accuracy: 0.7945 - val_classification_output_precision: 0.8093 - val_classification_output_recall: 0.7772
Epoch 17/20
98/98 [==============================] - 29s 301ms/step - loss: 0.3850 - classification_output_loss: 0.3850 - classification_output_accuracy: 0.8360 - classification_output_precision: 0.8443 - classification_output_recall: 0.8250 - val_loss: 0.4640 - val_classification_output_loss: 0.4640 - val_classification_output_accuracy: 0.7845 - val_classification_output_precision: 0.8053 - val_classification_output_recall: 0.7574
Epoch 18/20
98/98 [==============================] - 29s 300ms/step - loss: 0.3785 - classification_output_loss: 0.3785 - classification_output_accuracy: 0.8417 - classification_output_precision: 0.8506 - classification_output_recall: 0.8300 - val_loss: 0.4578 - val_classification_output_loss: 0.4578 - val_classification_output_accuracy: 0.7945 - val_classification_output_precision: 0.8000 - val_classification_output_recall: 0.7921
Epoch 19/20
98/98 [==============================] - 29s 300ms/step - loss: 0.3700 - classification_output_loss: 0.3700 - classification_output_accuracy: 0.8478 - classification_output_precision: 0.8575 - classification_output_recall: 0.8351 - val_loss: 0.4562 - val_classification_output_loss: 0.4562 - val_classification_output_accuracy: 0.7970 - val_classification_output_precision: 0.8071 - val_classification_output_recall: 0.7871
Epoch 20/20
98/98 [==============================] - 30s 302ms/step - loss: 0.3617 - classification_output_loss: 0.3617 - classification_output_accuracy: 0.8539 - classification_output_precision: 0.8683 - classification_output_recall: 0.8351 - val_loss: 0.4535 - val_classification_output_loss: 0.4535 - val_classification_output_accuracy: 0.8045 - val_classification_output_precision: 0.7952 - val_classification_output_recall: 0.8267
print("Extending Training to 35 Epochs...")

history_extension = unified_model.fit(
    train_ds_final,
    validation_data=val_ds_final,
    epochs=15, # This adds 15 to your current 20
    callbacks=[early_stop, checkpoint]
)
Extending Training to 35 Epochs...
Epoch 1/15
98/98 [==============================] - 30s 303ms/step - loss: 0.3549 - classification_output_loss: 0.3549 - classification_output_accuracy: 0.8577 - classification_output_precision: 0.8594 - classification_output_recall: 0.8561 - val_loss: 0.4516 - val_classification_output_loss: 0.4516 - val_classification_output_accuracy: 0.7995 - val_classification_output_precision: 0.7905 - val_classification_output_recall: 0.8218
Epoch 2/15
98/98 [==============================] - 29s 300ms/step - loss: 0.3478 - classification_output_loss: 0.3478 - classification_output_accuracy: 0.8631 - classification_output_precision: 0.8679 - classification_output_recall: 0.8574 - val_loss: 0.4515 - val_classification_output_loss: 0.4515 - val_classification_output_accuracy: 0.7945 - val_classification_output_precision: 0.7970 - val_classification_output_recall: 0.7970
Epoch 3/15
98/98 [==============================] - 29s 300ms/step - loss: 0.3406 - classification_output_loss: 0.3406 - classification_output_accuracy: 0.8676 - classification_output_precision: 0.8758 - classification_output_recall: 0.8574 - val_loss: 0.4484 - val_classification_output_loss: 0.4484 - val_classification_output_accuracy: 0.8045 - val_classification_output_precision: 0.8010 - val_classification_output_recall: 0.8168
Epoch 4/15
98/98 [==============================] - 29s 297ms/step - loss: 0.3338 - classification_output_loss: 0.3338 - classification_output_accuracy: 0.8708 - classification_output_precision: 0.8786 - classification_output_recall: 0.8612 - val_loss: 0.4487 - val_classification_output_loss: 0.4487 - val_classification_output_accuracy: 0.8020 - val_classification_output_precision: 0.8030 - val_classification_output_recall: 0.8069
Epoch 5/15
98/98 [==============================] - 29s 297ms/step - loss: 0.3277 - classification_output_loss: 0.3277 - classification_output_accuracy: 0.8724 - classification_output_precision: 0.8790 - classification_output_recall: 0.8644 - val_loss: 0.4486 - val_classification_output_loss: 0.4486 - val_classification_output_accuracy: 0.7970 - val_classification_output_precision: 0.7951 - val_classification_output_recall: 0.8069
Epoch 6/15
98/98 [==============================] - 96s 982ms/step - loss: 0.3211 - classification_output_loss: 0.3211 - classification_output_accuracy: 0.8768 - classification_output_precision: 0.8860 - classification_output_recall: 0.8657 - val_loss: 0.4473 - val_classification_output_loss: 0.4473 - val_classification_output_accuracy: 0.7995 - val_classification_output_precision: 0.7933 - val_classification_output_recall: 0.8168
Epoch 7/15
98/98 [==============================] - 30s 302ms/step - loss: 0.3151 - classification_output_loss: 0.3151 - classification_output_accuracy: 0.8819 - classification_output_precision: 0.8862 - classification_output_recall: 0.8771 - val_loss: 0.4476 - val_classification_output_loss: 0.4476 - val_classification_output_accuracy: 0.8070 - val_classification_output_precision: 0.8173 - val_classification_output_recall: 0.7970
Epoch 8/15
98/98 [==============================] - 30s 303ms/step - loss: 0.3090 - classification_output_loss: 0.3090 - classification_output_accuracy: 0.8829 - classification_output_precision: 0.8930 - classification_output_recall: 0.8708 - val_loss: 0.4469 - val_classification_output_loss: 0.4469 - val_classification_output_accuracy: 0.8070 - val_classification_output_precision: 0.8109 - val_classification_output_recall: 0.8069
Epoch 9/15
98/98 [==============================] - 29s 300ms/step - loss: 0.3027 - classification_output_loss: 0.3027 - classification_output_accuracy: 0.8886 - classification_output_precision: 0.8983 - classification_output_recall: 0.8771 - val_loss: 0.4463 - val_classification_output_loss: 0.4463 - val_classification_output_accuracy: 0.7920 - val_classification_output_precision: 0.7820 - val_classification_output_recall: 0.8168
Epoch 10/15
98/98 [==============================] - 32s 331ms/step - loss: 0.2966 - classification_output_loss: 0.2966 - classification_output_accuracy: 0.8976 - classification_output_precision: 0.9037 - classification_output_recall: 0.8905 - val_loss: 0.4447 - val_classification_output_loss: 0.4447 - val_classification_output_accuracy: 0.8045 - val_classification_output_precision: 0.8069 - val_classification_output_recall: 0.8069
Epoch 11/15
98/98 [==============================] - 37s 384ms/step - loss: 0.2916 - classification_output_loss: 0.2916 - classification_output_accuracy: 0.8950 - classification_output_precision: 0.9022 - classification_output_recall: 0.8867 - val_loss: 0.4452 - val_classification_output_loss: 0.4452 - val_classification_output_accuracy: 0.8020 - val_classification_output_precision: 0.8060 - val_classification_output_recall: 0.8020
Epoch 12/15
98/98 [==============================] - 39s 400ms/step - loss: 0.2860 - classification_output_loss: 0.2860 - classification_output_accuracy: 0.8989 - classification_output_precision: 0.9082 - classification_output_recall: 0.8880 - val_loss: 0.4459 - val_classification_output_loss: 0.4459 - val_classification_output_accuracy: 0.7995 - val_classification_output_precision: 0.7990 - val_classification_output_recall: 0.8069
Epoch 13/15
98/98 [==============================] - 38s 386ms/step - loss: 0.2806 - classification_output_loss: 0.2806 - classification_output_accuracy: 0.8985 - classification_output_precision: 0.9081 - classification_output_recall: 0.8873 - val_loss: 0.4461 - val_classification_output_loss: 0.4461 - val_classification_output_accuracy: 0.8045 - val_classification_output_precision: 0.8069 - val_classification_output_recall: 0.8069
Epoch 14/15
98/98 [==============================] - 38s 387ms/step - loss: 0.2751 - classification_output_loss: 0.2751 - classification_output_accuracy: 0.9030 - classification_output_precision: 0.9100 - classification_output_recall: 0.8950 - val_loss: 0.4452 - val_classification_output_loss: 0.4452 - val_classification_output_accuracy: 0.8020 - val_classification_output_precision: 0.8030 - val_classification_output_recall: 0.8069
Epoch 15/15
98/98 [==============================] - ETA: 0s - loss: 0.2697 - classification_output_loss: 0.2697 - classification_output_accuracy: 0.9068 - classification_output_precision: 0.9123 - classification_output_recall: 0.9007Restoring model weights from the end of the best epoch: 10.
98/98 [==============================] - 39s 396ms/step - loss: 0.2697 - classification_output_loss: 0.2697 - classification_output_accuracy: 0.9068 - classification_output_precision: 0.9123 - classification_output_recall: 0.9007 - val_loss: 0.4484 - val_classification_output_loss: 0.4484 - val_classification_output_accuracy: 0.7945 - val_classification_output_precision: 0.8030 - val_classification_output_recall: 0.7871
Epoch 15: early stopping

import tensorflow as tf
import gc

# --- STEP 1: CLEAR MEMORY ---
print("Cleaning up GPU memory...")
tf.keras.backend.clear_session()
gc.collect()

# --- STEP 2: RELOAD DATA WITH SMALLER BATCH SIZE (Fixes OOM) ---
# We reduce BATCH_SIZE from 32 to 8 to fit the gradients in VRAM
NEW_BATCH_SIZE = 8 

# Re-wrap your existing datasets with the new batch size
# Note: This assumes train_ds and val_ds are your original tf.data.Dataset objects
train_ds_low_mem = train_ds.unbatch().batch(NEW_BATCH_SIZE).map(prepare_for_training)
val_ds_low_mem = val_ds.unbatch().batch(NEW_BATCH_SIZE).map(prepare_for_training)

# --- STEP 3: PARTIAL UNFREEZING ---
# We unfreeze only the last block of VGG-Face (the high-level features)
feature_extractor.trainable = True

# Freeze everything EXCEPT the last 4 layers of the backbone
# This significantly reduces memory usage compared to unfreezing everything
for layer in feature_extractor.layers[:-4]:
    layer.trainable = False

print(f"Backbone unfrozen for final {len(feature_extractor.layers[-4:])} layers.")

# --- STEP 4: RE-COMPILE WITH ULTRA-LOW LEARNING RATE ---
# Use 1e-5 (0.00001) to delicately tune the pre-trained weights
unified_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss={'classification_output': 'binary_crossentropy'},
    metrics={'classification_output': ['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]}
)

# --- STEP 5: RUN FINE-TUNING ---
print("Starting Phase 2: Specialist Fine-Tuning...")
history_fine = unified_model.fit(
    train_ds_low_mem,
    validation_data=val_ds_low_mem,
    epochs=10, 
    callbacks=[early_stop, checkpoint]
)
Cleaning up GPU memory...
Backbone unfrozen for final 4 layers.
Starting Phase 2: Specialist Fine-Tuning...
Epoch 1/10
392/392 [==============================] - 101s 246ms/step - loss: 0.3045 - classification_output_loss: 0.3045 - classification_output_accuracy: 0.8615 - classification_output_precision: 0.8708 - classification_output_recall: 0.8498 - val_loss: 0.3999 - val_classification_output_loss: 0.3999 - val_classification_output_accuracy: 0.8120 - val_classification_output_precision: 0.8191 - val_classification_output_recall: 0.8069
Epoch 2/10
392/392 [==============================] - 88s 224ms/step - loss: 0.0795 - classification_output_loss: 0.0795 - classification_output_accuracy: 0.9831 - classification_output_precision: 0.9897 - classification_output_recall: 0.9764 - val_loss: 0.3656 - val_classification_output_loss: 0.3656 - val_classification_output_accuracy: 0.8596 - val_classification_output_precision: 0.8614 - val_classification_output_recall: 0.8614
Epoch 3/10
392/392 [==============================] - 86s 219ms/step - loss: 0.0263 - classification_output_loss: 0.0263 - classification_output_accuracy: 0.9978 - classification_output_precision: 0.9987 - classification_output_recall: 0.9968 - val_loss: 0.3865 - val_classification_output_loss: 0.3865 - val_classification_output_accuracy: 0.8546 - val_classification_output_precision: 0.8600 - val_classification_output_recall: 0.8515
Epoch 4/10
392/392 [==============================] - 83s 213ms/step - loss: 0.0109 - classification_output_loss: 0.0109 - classification_output_accuracy: 0.9994 - classification_output_precision: 0.9994 - classification_output_recall: 0.9994 - val_loss: 0.4336 - val_classification_output_loss: 0.4336 - val_classification_output_accuracy: 0.8521 - val_classification_output_precision: 0.8629 - val_classification_output_recall: 0.8416
Epoch 5/10
392/392 [==============================] - 126s 322ms/step - loss: 0.0113 - classification_output_loss: 0.0113 - classification_output_accuracy: 0.9981 - classification_output_precision: 0.9981 - classification_output_recall: 0.9981 - val_loss: 0.4601 - val_classification_output_loss: 0.4601 - val_classification_output_accuracy: 0.8571 - val_classification_output_precision: 0.8796 - val_classification_output_recall: 0.8317
Epoch 6/10
392/392 [==============================] - 75s 192ms/step - loss: 0.0052 - classification_output_loss: 0.0052 - classification_output_accuracy: 0.9994 - classification_output_precision: 0.9994 - classification_output_recall: 0.9994 - val_loss: 0.4749 - val_classification_output_loss: 0.4749 - val_classification_output_accuracy: 0.8446 - val_classification_output_precision: 0.8608 - val_classification_output_recall: 0.8267
Epoch 7/10
392/392 [==============================] - ETA: 0s - loss: 0.0016 - classification_output_loss: 0.0016 - classification_output_accuracy: 1.0000 - classification_output_precision: 1.0000 - classification_output_recall: 1.0000Restoring model weights from the end of the best epoch: 2.
392/392 [==============================] - 82s 209ms/step - loss: 0.0016 - classification_output_loss: 0.0016 - classification_output_accuracy: 1.0000 - classification_output_precision: 1.0000 - classification_output_recall: 1.0000 - val_loss: 0.5082 - val_classification_output_loss: 0.5082 - val_classification_output_accuracy: 0.8596 - val_classification_output_precision: 0.8763 - val_classification_output_recall: 0.8416
Epoch 7: early stopping

import tensorflow as tf
import gc
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# 1. Clear GPU memory to make room
tf.keras.backend.clear_session()
gc.collect()

# 2. Re-batch the Test Dataset to a smaller size (8)
# This assumes 'test_ds' is your original test dataset variable
test_ds_low_mem = test_ds.unbatch().batch(8).map(prepare_for_training)

print("Evaluating on Test Dataset with low-memory configuration...")

# 3. Perform Evaluation
evaluation_results = unified_model.evaluate(test_ds_low_mem)
print(f"\nTest Accuracy: {evaluation_results[1]*100:.2f}%")
print(f"Test Precision: {evaluation_results[2]*100:.2f}%")
print(f"Test Recall: {evaluation_results[3]*100:.2f}%")

# 4. Get Predictions for the Confusion Matrix
y_true = []
y_pred = []

print("\nGenerating Confusion Matrix data...")
for images, labels in test_ds_low_mem:
    true_labels = labels['classification_output'].numpy().flatten()
    # We use verbose=0 to keep the console clean
    preds, _ = unified_model.predict(images, verbose=0)
    
    y_true.extend(true_labels)
    y_pred.extend((preds > 0.5).astype(int).flatten())

# 5. Visualize
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Non-Autistic', 'Autistic'], 
            yticklabels=['Non-Autistic', 'Autistic'])
plt.title('Final Confusion Matrix (Test Set)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print("\n--- Final Classification Report ---")
print(classification_report(y_true, y_pred, target_names=['Non-Autistic', 'Autistic']))

Evaluating on Test Dataset with low-memory configuration...
48/48 [==============================] - 7s 144ms/step - loss: 0.4446 - classification_output_loss: 0.4446 - classification_output_accuracy: 0.8281 - classification_output_precision: 0.8041 - classification_output_recall: 0.8478

Test Accuracy: 44.46%
Test Precision: 82.81%
Test Recall: 80.41%

Generating Confusion Matrix data...
--- Final Classification Report ---
              precision    recall  f1-score   support

Non-Autistic       0.85      0.81      0.83       200
    Autistic       0.80      0.85      0.83       184

    accuracy                           0.83       384
   macro avg       0.83      0.83      0.83       384
weighted avg       0.83      0.83      0.83       384
from sklearn.manifold import TSNE
import pandas as pd

# 1. Extract features from the test set
print("Extracting feature vectors for visualization...")
all_features = []
all_labels = []

for images, labels in test_ds_low_mem:
    # Use the second output of our unified model
    _, features = unified_model.predict(images, verbose=0)
    all_features.append(features)
    all_labels.extend(labels['classification_output'].numpy().flatten())

all_features = np.concatenate(all_features, axis=0)
all_labels = np.array(all_labels)

# 2. Run t-SNE (Reduces 256D down to 2D for plotting)
tsne = TSNE(n_components=2, random_state=SEED)
low_dim_features = tsne.fit_transform(all_features)

# 3. Plot the Clusters
plt.figure(figsize=(10, 7))
df = pd.DataFrame({
    'x': low_dim_features[:, 0],
    'y': low_dim_features[:, 1],
    'Label': ['Autistic' if l == 1 else 'Non-Autistic' for l in all_labels]
})

sns.scatterplot(data=df, x='x', y='y', hue='Label', palette='viridis', alpha=0.7)
plt.title('t-SNE Visualization: How the Model Groups Facial Features')
plt.show()

